# Recovery of $\Delta\eta = \eta_{\text{I}} - \eta_{\text{II}}$ under different rating distributions

Fix $\eta_{\text{I}} = 0.3$ and vary $\Delta\eta = \eta_{\text{I}} - \eta_{\text{II}}$ (so $\eta_{\text{II}} = 0.3 - \Delta\eta$).

- Population I: $r \in \{1, 2, \ldots, 9\}$ (wide range)
- Population II: $r \in \{4, 5, 6\}$ (narrow range)

For each $\Delta\eta$, estimate $(\eta_{\text{I}}, \eta_{\text{II}})$ independently for each subject,
then compute $\widehat{\Delta\eta} = \hat\eta_{\text{I}} - \hat\eta_{\text{II}}$.

**Expected result**: MLE $\widehat{\Delta\eta}$ lies on the identity line.
TADA $\widehat{\Delta\eta}$ is shifted upward (TADA underestimates $\eta_{\text{II}}$ more than $\eta_{\text{I}}$
due to the narrow rating range), producing a systematic positive bias in $\widehat{\Delta\eta}$.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from scipy.optimize import minimize, Bounds, LinearConstraint
from scipy.stats import chi2
import warnings
import time
import pickle
import glob

from efpt import aDDModel
from efpt.cython.batch import compute_addm_nll, compute_tada_mean_nll

import sys
sys.path.insert(0, "../..")
from shared import (extract_data, fit_subject, subject_sum_nll,
                     fit_equal_eta, greater_lrt, less_lrt, setup_latex_matplotlib)
setup_latex_matplotlib()

warnings.filterwarnings("ignore", message=r"delta_grad == 0\.0.*", category=UserWarning)

In [2]:
# Parameters
ETA_I = 0.3
SHARED_PARAMS = dict(kappa=0.5, sigma=1.0, a=2.0, b=0.0, x0=0.0)

# Delta values: eta_I - eta_II
# Negative delta means eta_I < eta_II, positive means eta_I > eta_II
DELTA_VALUES = np.linspace(-0.1, 0.1, 11) 
# Rating ranges
R_RANGE_I = (1, 9)    # wide
R_RANGE_II = (4, 6)    # narrow

# Fixation distribution
GAMMA_SHAPE = 4.0
GAMMA_SCALE = 0.1

# Experiment settings
N_TRIALS = 2000
N_REPLICATIONS = 50
N_THREADS = -1

print(f"eta_I = {ETA_I} (fixed)")
print(f"delta = eta_I - eta_II in {DELTA_VALUES}")
print(f"eta_II = {ETA_I} - delta in [{ETA_I - DELTA_VALUES[-1]:.2f}, {ETA_I - DELTA_VALUES[0]:.2f}]")
print(f"Population I: r_range={R_RANGE_I}, Population II: r_range={R_RANGE_II}")
print(f"{N_TRIALS} trials/subject, {N_REPLICATIONS} replications per delta")

eta_I = 0.3 (fixed)
delta = eta_I - eta_II in [-0.1  -0.08 -0.06 -0.04 -0.02  0.    0.02  0.04  0.06  0.08  0.1 ]
eta_II = 0.3 - delta in [0.20, 0.40]
Population I: r_range=(1, 9), Population II: r_range=(4, 6)
2000 trials/subject, 50 replications per delta


In [3]:
# Helper functions imported from shared.py:
#   extract_data, fit_subject, subject_sum_nll,
#   fit_equal_eta, greater_lrt, less_lrt

In [4]:
# Main loop
rng = np.random.default_rng(42)

results = {"mle": {}, "tada": {}}
for mode in ["mle", "tada"]:
    results[mode] = {d: [] for d in DELTA_VALUES}

start_time = time.time()
for delta in DELTA_VALUES:
    eta_II = ETA_I - delta
    model_I = aDDModel(eta=ETA_I, **SHARED_PARAMS)
    model_II = aDDModel(eta=eta_II, **SHARED_PARAMS)

    for rep in range(N_REPLICATIONS):
        exp_I = model_I.generate_experiment(
            n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
            r_range=R_RANGE_I, n_threads=N_THREADS, rng=rng,
        )
        exp_II = model_II.generate_experiment(
            n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
            r_range=R_RANGE_II, n_threads=N_THREADS, rng=rng,
        )
        data_I = extract_data(exp_I)
        data_II = extract_data(exp_II)

        for mode in ["mle", "tada"]:
            res_I = fit_subject(data_I, mode, n_threads=N_THREADS)
            res_II = fit_subject(data_II, mode, n_threads=N_THREADS)
            delta_hat = res_I.x[0] - res_II.x[0]
            results[mode][delta].append(delta_hat)

    elapsed = time.time() - start_time
    mle_mean = np.mean(results["mle"][delta])
    tada_mean = np.mean(results["tada"][delta])
    print(f"delta={delta:+.3f} (eta_II={eta_II:.3f}): "
          f"MLE delta_hat={mle_mean:+.4f}, TADA delta_hat={tada_mean:+.4f} "
          f"[{elapsed:.0f}s]")

print(f"\nTotal time: {time.time() - start_time:.0f}s")

fname = "varying_delta_result_" + time.strftime("%Y%m%d-%H%M%S") + ".pkl"
with open(fname, "wb") as f:
    pickle.dump({"DELTA_VALUES": DELTA_VALUES, "results": results,
                 "ETA_I": ETA_I, "N_TRIALS": N_TRIALS, "N_REPLICATIONS": N_REPLICATIONS}, f)
print(f"Saved to {fname}")

delta=-0.100 (eta_II=0.400): MLE delta_hat=-0.0993, TADA delta_hat=-0.0139 [305s]
delta=-0.080 (eta_II=0.380): MLE delta_hat=-0.0858, TADA delta_hat=-0.0013 [727s]
delta=-0.060 (eta_II=0.360): MLE delta_hat=-0.0604, TADA delta_hat=+0.0200 [1003s]
delta=-0.040 (eta_II=0.340): MLE delta_hat=-0.0382, TADA delta_hat=+0.0491 [1287s]
delta=-0.020 (eta_II=0.320): MLE delta_hat=-0.0212, TADA delta_hat=+0.0607 [1559s]
delta=+0.000 (eta_II=0.300): MLE delta_hat=+0.0005, TADA delta_hat=+0.0833 [1882s]
delta=+0.020 (eta_II=0.280): MLE delta_hat=+0.0238, TADA delta_hat=+0.1079 [2223s]
delta=+0.040 (eta_II=0.260): MLE delta_hat=+0.0416, TADA delta_hat=+0.1201 [2494s]
delta=+0.060 (eta_II=0.240): MLE delta_hat=+0.0619, TADA delta_hat=+0.1405 [2768s]
delta=+0.080 (eta_II=0.220): MLE delta_hat=+0.0844, TADA delta_hat=+0.1609 [3004s]
delta=+0.100 (eta_II=0.200): MLE delta_hat=+0.1009, TADA delta_hat=+0.1751 [3245s]

Total time: 3245s
Saved to varying_delta_result_20260522-163735.pkl


In [3]:
# Load results (if necessary)
result_files = sorted(glob.glob("varying_delta_result_*.pkl"))
with open(result_files[-1], "rb") as f:
    data = pickle.load(f)
DELTA_VALUES = data["DELTA_VALUES"]
results = data["results"]
print(f"Loaded from {result_files[-1]}")

Loaded from varying_delta_result_20260522-163735.pkl


In [7]:
# Plot: true delta vs estimated delta
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

ax.plot([-0.3, 0.3], [-0.3, 0.3], "k-", lw=1, label="identity function")

for mode, color, marker, label in [("mle", "blue", "o", rf"$\widehat{{\Delta \eta}}_{{10000}}^{{\mathrm{{ML}}}}$"),
                                    ("tada", "red", "x", rf"$\widehat{{\Delta \eta}}_{{10000}}^{{\mathrm{{TADA}}}}$")]:
    means = [np.mean(results[mode][d]) for d in DELTA_VALUES]
    stds = [np.std(results[mode][d]) for d in DELTA_VALUES]
    ax.errorbar(DELTA_VALUES, means, yerr=stds, fmt=marker, color=color,
                capsize=5, ms=8, lw=1.5, label=label)


ax.axhline(0, color="gray", ls=":", alpha=0.5)
ax.axvline(0, color="gray", ls=":", alpha=0.5)

ax.set_xlabel(r"True $\Delta\eta = \eta_{\text{I}} - \eta_{\text{II}}$", fontsize=20)
ax.set_ylabel(r"Estimated $\widehat{\Delta\eta} = \widehat\eta_{\text{I}} - \widehat\eta_{\text{II}}$", fontsize=20)
# ax.set_title(rf"$\eta_{\text{I}} = {ETA_I}$ fixed, Population I: $r \in {R_RANGE_I}$, Population II: $r \in {R_RANGE_II}$",
ax.legend(fontsize=15)
ax.tick_params(axis="both", labelsize=15)
ax.set_xlim(-0.12, 0.12)
ax.set_ylim(-0.26, 0.2)
# ax.set_aspect("equal")

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# Shade quadrant II: x < 0, y > 0
ax.add_patch(Rectangle(
    (xmin, 0),          # lower-left corner
    0 - xmin,           # width
    ymax - 0,           # height
    facecolor="gray",
    alpha=0.12,
    edgecolor="none",
    zorder=0
))

# Shade quadrant IV: x > 0, y < 0
ax.add_patch(Rectangle(
    (0, ymin),
    xmax - 0,
    0 - ymin,
    facecolor="gray",
    alpha=0.12,
    edgecolor="none",
    zorder=0
))


plt.tight_layout()
plt.savefig("population_with_different_rating_distributions.png", bbox_inches="tight")
plt.show()

## One-sided LRT at $\Delta\eta = -0.05$: Type I error

Truth: $\eta_{\text{I}} = 0.30$, $\eta_{\text{II}} = 0.35$, so $\eta_{\text{I}} < \eta_{\text{II}}$ and $H_0: \eta_{\text{I}} \le \eta_{\text{II}}$ is **true**.

Rejections are **Type I errors**.

TADA's positive bias in $\widehat{\Delta\eta}$ (from the plot above) makes it estimate $\hat\eta_{\text{I}} > \hat\eta_{\text{II}}$, so TADA's sign check passes and it falsely rejects $H_0$.

In [7]:
DELTA_TEST = -0.05
ETA_II_TEST = ETA_I - DELTA_TEST
ALPHA = 0.05
CRITICAL_ONE_SIDED = chi2.ppf(1 - 2 * ALPHA, df=1)
N_REPS_LRT = 200

model_I = aDDModel(eta=ETA_I, **SHARED_PARAMS)
model_II = aDDModel(eta=ETA_II_TEST, **SHARED_PARAMS)
rng_lrt = np.random.default_rng(123)

greater_results = {"mle": [], "tada": []}

print(f"One-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II")
print(f"Truth: eta_I={ETA_I}, eta_II={ETA_II_TEST} (delta={DELTA_TEST}), H0 is TRUE")
print(f"Rejections are TYPE I ERRORS")
print(f"Critical value (alpha={ALPHA}): {CRITICAL_ONE_SIDED:.4f}")
print()

start_time = time.time()
for rep in range(N_REPS_LRT):
    exp_I = model_I.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE_I, n_threads=N_THREADS, rng=rng_lrt,
    )
    exp_II = model_II.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE_II, n_threads=N_THREADS, rng=rng_lrt,
    )
    dI = extract_data(exp_I)
    dII = extract_data(exp_II)

    for mode in ["mle", "tada"]:
        Lambda, reject, eta_I_hat, eta_II_hat = greater_lrt(
            dI, dII, mode, CRITICAL_ONE_SIDED, n_threads=N_THREADS)
        greater_results[mode].append({
            "Lambda": Lambda, "reject": reject,
            "eta_I": eta_I_hat, "eta_II": eta_II_hat,
        })

    if (rep + 1) % 20 == 0 or rep == 0:
        elapsed = time.time() - start_time
        mle_err = np.mean([r["reject"] for r in greater_results["mle"]])
        tada_err = np.mean([r["reject"] for r in greater_results["tada"]])
        print(f"Rep {rep+1:3d}/{N_REPS_LRT}: "
              f"MLE Type I error={mle_err:.3f}, TADA Type I error={tada_err:.3f} "
              f"[{elapsed:.0f}s]")

print(f"\nTotal time: {time.time() - start_time:.0f}s")

fname = "greater_lrt_result_" + time.strftime("%Y%m%d-%H%M%S") + ".pkl"
with open(fname, "wb") as f:
    pickle.dump(greater_results, f)
print(f"Saved to {fname}")

One-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II
Truth: eta_I=0.3, eta_II=0.35 (delta=-0.05), H0 is TRUE
Rejections are TYPE I ERRORS
Critical value (alpha=0.05): 2.7055

Rep   1/200: MLE Type I error=0.000, TADA Type I error=1.000 [7s]
Rep  20/200: MLE Type I error=0.000, TADA Type I error=0.400 [141s]
Rep  40/200: MLE Type I error=0.000, TADA Type I error=0.475 [309s]
Rep  60/200: MLE Type I error=0.000, TADA Type I error=0.467 [465s]
Rep  80/200: MLE Type I error=0.000, TADA Type I error=0.412 [622s]
Rep 100/200: MLE Type I error=0.000, TADA Type I error=0.400 [770s]
Rep 120/200: MLE Type I error=0.000, TADA Type I error=0.383 [987s]
Rep 140/200: MLE Type I error=0.000, TADA Type I error=0.364 [1161s]
Rep 160/200: MLE Type I error=0.000, TADA Type I error=0.344 [1314s]
Rep 180/200: MLE Type I error=0.000, TADA Type I error=0.350 [1453s]
Rep 200/200: MLE Type I error=0.000, TADA Type I error=0.345 [1604s]

Total time: 1604s
Saved to greater_lrt_result_20260522-170419.pkl


In [8]:
greater_files = sorted(glob.glob("greater_lrt_result_*.pkl"))
with open(greater_files[-1], "rb") as f:
    greater_results = pickle.load(f)
print(f"Loaded from {greater_files[-1]}")

print(f"\nOne-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II")
print(f"Truth: eta_I={ETA_I}, eta_II={ETA_II_TEST} (delta={DELTA_TEST}), H0 is TRUE")
print(f"alpha={ALPHA}, critical value={CRITICAL_ONE_SIDED:.4f}\n")

for mode in ["mle", "tada"]:
    etas_I = [r["eta_I"] for r in greater_results[mode]]
    etas_II = [r["eta_II"] for r in greater_results[mode]]
    rejections = [r["reject"] for r in greater_results[mode]]
    wrong_dir = [r["eta_I"] > r["eta_II"] for r in greater_results[mode]]
    n_rep = len(rejections)
    print(f"{mode.upper():4s}:")
    print(f"  mean eta_I = {np.mean(etas_I):.4f}, mean eta_II = {np.mean(etas_II):.4f}")
    print(f"  wrong direction (eta_I_hat > eta_II_hat): {sum(wrong_dir)}/{n_rep} ({np.mean(wrong_dir):.1%})")
    print(f"  reject H0 (Type I error): {sum(rejections)}/{n_rep} ({np.mean(rejections):.1%})")
    print()

Loaded from greater_lrt_result_20260522-170419.pkl

One-sided LRT: H0: eta_I <= eta_II  vs  H1: eta_I > eta_II
Truth: eta_I=0.3, eta_II=0.35 (delta=-0.05), H0 is TRUE
alpha=0.05, critical value=2.7055

MLE :
  mean eta_I = 0.3001, mean eta_II = 0.3496
  wrong direction (eta_I_hat > eta_II_hat): 3/200 (1.5%)
  reject H0 (Type I error): 0/200 (0.0%)

TADA:
  mean eta_I = 0.1967, mean eta_II = 0.1627
  wrong direction (eta_I_hat > eta_II_hat): 181/200 (90.5%)
  reject H0 (Type I error): 69/200 (34.5%)



## One-sided LRT for $H_0: \eta_{\text{I}} \ge \eta_{\text{II}}$ at $\Delta\eta = -0.05$: Power

Same truth: $\eta_{\text{I}} = 0.30$, $\eta_{\text{II}} = 0.35$. Now test the **opposite** direction:

$$H_0: \eta_{\text{I}} \ge \eta_{\text{II}} \qquad \text{vs} \qquad H_1: \eta_{\text{I}} < \eta_{\text{II}}$$

$H_0$ is **false** (since $0.30 < 0.35$). Rejections measure **power**.

MLE should have some power to detect the true ordering. TADA's positive bias in $\widehat{\Delta\eta}$ makes it overestimate $\eta_{\text{I}} - \eta_{\text{II}}$, so the sign check $\hat\eta_{\text{I}} < \hat\eta_{\text{II}}$ rarely passes — TADA has low power to detect the correct ordering.

In [9]:
model_I_less = aDDModel(eta=ETA_I, **SHARED_PARAMS)
model_II_less = aDDModel(eta=ETA_II_TEST, **SHARED_PARAMS)
rng_less = np.random.default_rng(456)

less_results = {"mle": [], "tada": []}

print(f"One-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II")
print(f"Truth: eta_I={ETA_I}, eta_II={ETA_II_TEST} (delta={DELTA_TEST}), H0 is FALSE")
print(f"Critical value (alpha={ALPHA}): {CRITICAL_ONE_SIDED:.4f}")
print()

start_time = time.time()
for rep in range(N_REPS_LRT):
    exp_I = model_I_less.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE_I, n_threads=N_THREADS, rng=rng_less,
    )
    exp_II = model_II_less.generate_experiment(
        n_trials=N_TRIALS, gamma_shape=GAMMA_SHAPE, gamma_scale=GAMMA_SCALE,
        r_range=R_RANGE_II, n_threads=N_THREADS, rng=rng_less,
    )
    dI = extract_data(exp_I)
    dII = extract_data(exp_II)

    for mode in ["mle", "tada"]:
        Lambda, reject, eta_I_hat, eta_II_hat = less_lrt(
            dI, dII, mode, CRITICAL_ONE_SIDED, n_threads=N_THREADS)
        less_results[mode].append({
            "Lambda": Lambda, "reject": reject,
            "eta_I": eta_I_hat, "eta_II": eta_II_hat,
        })

    if (rep + 1) % 20 == 0 or rep == 0:
        elapsed = time.time() - start_time
        mle_power = np.mean([r["reject"] for r in less_results["mle"]])
        tada_power = np.mean([r["reject"] for r in less_results["tada"]])
        print(f"Rep {rep+1:3d}/{N_REPS_LRT}: "
              f"MLE power={mle_power:.3f}, TADA power={tada_power:.3f} "
              f"[{elapsed:.0f}s]")

print(f"\nTotal time: {time.time() - start_time:.0f}s")

fname = "less_lrt_result_" + time.strftime("%Y%m%d-%H%M%S") + ".pkl"
with open(fname, "wb") as f:
    pickle.dump(less_results, f)
print(f"Saved to {fname}")

One-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II
Truth: eta_I=0.3, eta_II=0.35 (delta=-0.05), H0 is FALSE
Critical value (alpha=0.05): 2.7055

Rep   1/200: MLE power=1.000, TADA power=0.000 [27s]
Rep  20/200: MLE power=0.700, TADA power=0.000 [511s]
Rep  40/200: MLE power=0.700, TADA power=0.000 [923s]
Rep  60/200: MLE power=0.717, TADA power=0.000 [1382s]
Rep  80/200: MLE power=0.700, TADA power=0.000 [1823s]
Rep 100/200: MLE power=0.690, TADA power=0.000 [2283s]
Rep 120/200: MLE power=0.683, TADA power=0.000 [2767s]
Rep 140/200: MLE power=0.707, TADA power=0.007 [3210s]
Rep 160/200: MLE power=0.706, TADA power=0.006 [3641s]
Rep 180/200: MLE power=0.694, TADA power=0.006 [4107s]
Rep 200/200: MLE power=0.700, TADA power=0.005 [4540s]

Total time: 4540s
Saved to less_lrt_result_20260522-181959.pkl


In [10]:
less_files = sorted(glob.glob("less_lrt_result_*.pkl"))
with open(less_files[-1], "rb") as f:
    less_results = pickle.load(f)
print(f"Loaded from {less_files[-1]}")

print(f"\nOne-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II")
print(f"Truth: eta_I={ETA_I} < eta_II={ETA_II_TEST}, so H0 is FALSE")
print(f"alpha={ALPHA}, critical value={CRITICAL_ONE_SIDED:.4f}\n")

for mode in ["mle", "tada"]:
    etas_I = [r["eta_I"] for r in less_results[mode]]
    etas_II = [r["eta_II"] for r in less_results[mode]]
    rejections = [r["reject"] for r in less_results[mode]]
    correct_dir = [r["eta_I"] < r["eta_II"] for r in less_results[mode]]
    n_rep = len(rejections)
    print(f"{mode.upper():4s}:")
    print(f"  mean eta_I = {np.mean(etas_I):.4f}, mean eta_II = {np.mean(etas_II):.4f}")
    print(f"  correct direction (eta_I_hat < eta_II_hat): {sum(correct_dir)}/{n_rep} ({np.mean(correct_dir):.1%})")
    print(f"  reject H0 (power): {sum(rejections)}/{n_rep} ({np.mean(rejections):.1%})")
    print()

Loaded from less_lrt_result_20260522-181959.pkl

One-sided LRT: H0: eta_I >= eta_II  vs  H1: eta_I < eta_II
Truth: eta_I=0.3 < eta_II=0.35, so H0 is FALSE
alpha=0.05, critical value=2.7055

MLE :
  mean eta_I = 0.2995, mean eta_II = 0.3491
  correct direction (eta_I_hat < eta_II_hat): 196/200 (98.0%)
  reject H0 (power): 140/200 (70.0%)

TADA:
  mean eta_I = 0.1962, mean eta_II = 0.1637
  correct direction (eta_I_hat < eta_II_hat): 21/200 (10.5%)
  reject H0 (power): 1/200 (0.5%)



## Distribution of the one-sided LRT statistic $\Lambda_+$

Under $H_0: \eta_{\text{I}} \le \eta_{\text{II}}$ (true, strictly), the MLE-based $\Lambda_+$ conditional on $\Lambda_+ > 0$ should approximately follow $\chi^2_1$. Since $H_0$ holds strictly, $P(\Lambda_+ = 0)$ should be $> 50\%$ for MLE.

The TADA-based $\Lambda_+$ is a pseudo-LRT: the $\chi^2_1$ calibration does not hold because TADA is inconsistent. Its positive bias in $\widehat{\Delta\eta}$ makes the sign check pass too often, inflating Type I error.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_grid = np.linspace(0.01, 15, 300)
chi2_pdf = chi2.pdf(x_grid, df=1)

for ax, mode, title in zip(axes, ["mle", "tada"], ["MLE (correct)", "TADA (inconsistent)"]):
    lambdas = np.array([r["Lambda"] for r in greater_results[mode]])
    rej_rate = np.mean([r["reject"] for r in greater_results[mode]])
    n_zero = np.sum(lambdas == 0.0)
    n_total = len(lambdas)
    frac_zero = n_zero / n_total

    lambdas_pos = lambdas[lambdas > 0]
    if len(lambdas_pos) > 0:
        ax.hist(lambdas_pos, bins=25, density=True, alpha=0.7, color="steelblue",
                label=rf"$\Lambda_+ \mid \Lambda_+ > 0$ ({len(lambdas_pos)}/{n_total} reps)")

    ax.plot(x_grid, chi2_pdf, "r-", lw=2, label=r"$\chi^2_1$ density")
    ax.axvline(CRITICAL_ONE_SIDED, color="k", ls="--", lw=1.5,
               label=rf"critical value = {CRITICAL_ONE_SIDED:.2f}")

    ax.set_xlabel(r"$\Lambda_+$", fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.set_title(f"{title}\n"
                 rf"$P(\Lambda_+=0)$: {frac_zero:.0%}, "
                 f"Type I error: {rej_rate:.1%} (nominal {ALPHA:.0%})",
                 fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlim(0, max(15, np.percentile(lambdas, 99) if lambdas.max() > 0 else 15))

plt.tight_layout()
plt.show()